In [1]:
using Integrals
using JLD2

In [2]:
rtol = 1e-16
atol = 1e-16
imax = 1e6;

In [3]:
function L₂(x::AbstractVector{<:Real})
    A, B, logH = x
    H = 10^logH
    hd = H * log(10.0)

    # Auxilary variables for f, g and its derivatives
    a(u) = u+H
    b(u) = u-H
    c(u) = (b(u) - a(u)*exp(-2u))
    d(u) = b(u)^2 - a(u)*b(u)*exp(-2u)
    e(u) = b(u)^2 + d(u)
    dH(u) = -2*(b(u) - H*exp(-2u))
    
    f(u)   = a(u) / c(u)
    f3(u)  = 2 / c(u)^2
    f33(u) = 4*(1 + exp(-2u)) / c(u)^3
    
    g(u)   = a(u)^2 / d(u)
    g3(u)  = 2*a(u) * (b(u)^2 + d(u)) / (b(u) * d(u)^2)
    g33(u) = 2*(-2a(u)*b(u)*e(u)*dH(u) + a(u)*d(u)*e(u) + b(u)*d(u)*(e(u) - a(u)*(2b(u)-dH(u)))) / (b(u)^2 * d(u)^3)
    
    p(u)   = exp(-u*(2+B))
    p2(u)  = -exp(-u*(2+B))
    p22(u) = u*exp(-u*(2+B))
    
    q(u)   = exp(-u*(4-B))
    q2(u)  = exp(-u*(4-B))
    q22(u) = u*exp(-u*(4-B))
    
    r(u)   = cos(u*A)
    r1(u)  = -sin(u*A)
    r11(u) = -u*cos(u*A)
    
    h(u, _) = (f(u) * p(u) + g(u) * q(u)) * r(u) / u
    
    h1(u, _) = (f(u) * p(u) + g(u) * q(u)) * r1(u)
    h2(u, _) = (f(u) * p2(u) + g(u) * q2(u)) * r(u)
    h3(u, _) = (f3(u) * p(u) + g3(u) * q(u)) * r(u)
    
    h11(u, _) = (f(u) * p(u) + g(u) * q(u)) * r11(u)
    h12(u, _) = (f(u) * p2(u) + g(u) * q2(u)) * r1(u) * u
    h13(u, _) = (f3(u) * p(u) + g3(u) * q(u)) * r1(u) * u
    
    h21(u, _) = (f(u) * p2(u) + g(u) * q2(u)) * r1(u) * u
    h22(u, _) = (f(u) * p22(u) + g(u) * q22(u)) * r(u)
    h23(u, _) = (f3(u) * p2(u) + g3(u) * q2(u)) * r(u) * u
    
    h31(u, _) = (f3(u) * p(u) + g3(u) * q(u)) * r1(u) * u
    h32(u, _) = (f3(u) * p2(u) + g3(u) * q2(u)) * r(u) * u
    h33(u, _) = (f33(u) * p(u) + g33(u) * q(u)) * r(u)
    
    paths = [(0.0, H+im), (H+im, H+1.0), (H+1.0, Inf)]
    M = 0.0
    M1, M2, M3 = 0.0, 0.0, 0.0
    M11, M12, M13 = 0.0, 0.0, 0.0
    M21, M22, M23 = 0.0, 0.0, 0.0
    M31, M32, M33 = 0.0, 0.0, 0.0
    for path in paths
        IP = IntegralProblem(h, path)
        
        IP1 = IntegralProblem(h1, path)
        IP2 = IntegralProblem(h2, path)
        IP3 = IntegralProblem(h3, path)

        IP11 = IntegralProblem(h11, path)
        IP12 = IntegralProblem(h12, path)
        IP13 = IntegralProblem(h13, path)

        IP21 = IntegralProblem(h21, path)
        IP22 = IntegralProblem(h22, path)
        IP23 = IntegralProblem(h23, path)

        IP31 = IntegralProblem(h31, path)
        IP32 = IntegralProblem(h32, path)
        IP33 = IntegralProblem(h33, path)
        
        M += solve(IP, QuadGKJL(); reltol=rtol, abstol=atol, maxiters=imax).u
        
        M1 += solve(IP1, QuadGKJL(); reltol=rtol, abstol=atol, maxiters=imax).u
        M2 += solve(IP2, QuadGKJL(); reltol=rtol, abstol=atol, maxiters=imax).u
        M3 += solve(IP3, QuadGKJL(); reltol=rtol, abstol=atol, maxiters=imax).u

        M11 += solve(IP11, QuadGKJL(); reltol=rtol, abstol=atol, maxiters=imax).u
        M12 += solve(IP12, QuadGKJL(); reltol=rtol, abstol=atol, maxiters=imax).u
        M13 += solve(IP13, QuadGKJL(); reltol=rtol, abstol=atol, maxiters=imax).u

        M21 += solve(IP21, QuadGKJL(); reltol=rtol, abstol=atol, maxiters=imax).u
        M22 += solve(IP22, QuadGKJL(); reltol=rtol, abstol=atol, maxiters=imax).u
        M23 += solve(IP23, QuadGKJL(); reltol=rtol, abstol=atol, maxiters=imax).u

        M31 += solve(IP31, QuadGKJL(); reltol=rtol, abstol=atol, maxiters=imax).u
        M32 += solve(IP32, QuadGKJL(); reltol=rtol, abstol=atol, maxiters=imax).u
        M33 += solve(IP33, QuadGKJL(); reltol=rtol, abstol=atol, maxiters=imax).u
    end

    # M = real(M)
    # GM = [real(M1), real(M2), real(M3)*hd]
    # HM = [
    #     real(M11)    real(M12)    real(M13)*hd;
    #     real(M12)    real(M22)    real(M23)*hd;
    #     real(M13)*hd real(M23)*hd (real(M33)*hd + real(M3)*log(10))*hd;
    # ]

    M = real(M)
    GM = [real(M1), real(M2), real(M3)]
    HM = [
        real(M11)    real(M12)    real(M13);
        real(M12)    real(M22)    real(M23);
        real(M13)    real(M23)    real(M33);
    ]
    
    return M, GM, HM
end;

In [4]:
x = [0.25, 0.8, log10(1.1)]

y₂, ∇y₂, Hy₂ = L₂(x)

println(y₂)
println(∇y₂)
println(Hy₂)

0.43981437828913056
[0.09079883824328171, 0.7101391159785378, 0.0770000732549923]
[0.3525174506247106 0.09402034872793774 -0.14479634555981696; 0.09402034872793774 -0.3525174506247106 0.697096881028642; -0.14479634555981696 0.697096881028642 -1.5535874769762168]


In [4]:
Amin, Amax = 0.0, 0.5
Bmin, Bmax = 0.0, 2.0
logHmin, logHmax = -2.0, 2.0;

total_points = 100

xv = Matrix{Float64}(undef, total_points, 3)

xv[:, 1] .= Amin .+ rand(Float64, total_points) .* (Amax - Amin)
xv[:, 2] .= Bmin .+ rand(Float64, total_points) .* (Bmax - Bmin)
xv[:, 3] .= logHmin .+ rand(Float64, total_points) .* (logHmax - logHmin);

In [5]:
L = Vector{Float64}(undef, size(xv, 1))
∇L = Matrix{Float64}(undef, size(xv))
HL = Array{Float64, 3}(undef, size(xv)..., 3);

In [6]:
for i in axes(xv, 1)
    if mod(i, 100) == 0
        println("$i / $total_points")
    end
    
    L[i], ∇L[i, :], HL[i, :, :] = L₁(view(xv, i, :))
end

@save "l2points.jld2" xv L ∇L HL;

100 / 100
